# Modul 06: Analisis Regresi Linier Sederhana dan Berganda
**Mata Kuliah:** Statistika Komputasi  
**Dosen Pengampu:** Dr. Ridwan Ilyas, S.Kom., M.T.  
**Program Studi:** Teknik Informatika, Universitas Jenderal Achmad Yani (UNJANI) 2026  
**Lisensi:** Open Source (MIT)

---

## 📖 1. Analisis Regresi Linier Sederhana dan Berganda

Regresi Linier adalah algoritma pembelajaran terawasi (*Supervised Learning*) fundamental untuk memodelkan hubungan kuantitatif antara variabel target kontinu $Y$ dengan satu atau lebih prediktor $X$:
1. **Model Matematis OLS (*Ordinary Least Squares*)**:
   - $Y = eta_0 + eta_1 X_1 + eta_2 X_2 + \dots + eta_k X_k + \epsilon$
   - Prinsip OLS: Meminimalkan jumlah kuadrat galat residual $\sum e_i^2 = \sum (Y_i - \hat{Y}_i)^2$.
2. **Evaluasi Performa & Signifikansi**:
   - **Koefisien Determinasi ($R^2$)**: Proporsi variasi target yang berhasil dijelaskan oleh model prediktor ($0 \le R^2 \le 1$).
   - **Adjusted $R^2$**: Penyesuaian nilai $R^2$ terhadap jumlah fitur agar tidak terdistorsi penambahan prediktor tidak relevan.
   - **Uji F (Simultan)**: Menguji apakah seluruh prediktor secara bersama-sama berpengaruh signifikan ($H_0: eta_1 = eta_2 = \dots = 0$).
   - **Uji t (Parsial)**: Menguji signifikansi individual setiap bobot koefisien prediktor.


## 📊 2. Diagram Ilustrasi Konsep

![Ilustrasi Model Regresi Linier Berganda](images/img_06_linear_regression.png)

> **Deskripsi Visual Infografis 2D:**
> 1. **1. Property Price Regression (OLS Plane)**: Garis tren terbaik $Y = \beta_0 + \beta_1 X + \varepsilon$ memprediksi harga properti berdasarkan luas ($m^2$) dengan garis putus-putus galat residual $e = Y - \hat{Y}$.
> 2. **2. Model Explanatory Power & Significance**: Evaluasi model melalui **$R^2 = 0.880$ (88% variansi harga terjelaskan)** dan signifikansi global **$F\text{-Stat} = 124.5, p < 0.001$** dengan kemiringan slope +$250/$m^2$.



## 🔬 3. Studi Kasus & Penjelasan Langkah Komputasi

Studi kasus memodelkan estimasi biaya pengembangan proyek perangkat lunak (`04_software_cost_regression.csv`) berdasarkan ukuran kode sumber (KLOC), jumlah anggota tim teknis, dan durasi pengerjaan.

**Tahapan Komputasi:**
1. Membentuk model regresi OLS menggunakan pustaka `statsmodels.api`.
2. Menganalisis tabel ringkasan OLS: Koefisien $eta$, p-value uji t, nilai $F$-statistic, dan $R^2$.
3. Melakukan diagnostik sebaran residual untuk memeriksa ketiadaan pola galat non-linier.


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
df_cost = pd.read_csv("../datasets/04_software_cost_regression.csv")
print("Data biaya proyek software dimuat:", df_cost.shape)
display(df_cost.head())


## 💻 4. Eksekusi Komputasi Python & Estimasi Model OLS


In [ ]:
# 1. Menentukan Fitur Prediktor dan Target
X = df_cost[['lines_of_code_kloc', 'team_size_members', 'development_months']]
y = df_cost['project_cost_million_idr']

# Menambahkan konstanta intercept beta_0
X_const = sm.add_constant(X)
ols_model = sm.OLS(y, X_const).fit()

# Menampilkan Ringkasan Parameter
print("=== Ringkasan Model Regresi Linier Berganda ===")
print(ols_model.summary().tables[1])
print(f"R-squared: {ols_model.rsquared:.3f} | Adjusted R-squared: {ols_model.rsquared_adj:.3f}")
print(f"F-statistic: {ols_model.fvalue:.2f} (p-value: {ols_model.f_pvalue:.4e})")


In [ ]:
# 2. Diagnostik Residual dan Perbandingan Aktual vs Prediksi
df_cost['predicted_cost'] = ols_model.predict(X_const)
df_cost['residuals'] = ols_model.resid

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot Aktual vs Prediksi
axes[0].scatter(df_cost['project_cost_million_idr'], df_cost['predicted_cost'], color='#1A365D', alpha=0.8)
axes[0].plot([df_cost['project_cost_million_idr'].min(), df_cost['project_cost_million_idr'].max()],
             [df_cost['project_cost_million_idr'].min(), df_cost['project_cost_million_idr'].max()],
             color='#EA580C', linestyle='--', lw=2, label='Garis Identitas Sempurna (1:1)')
axes[0].set_title('Aktual vs. Prediksi Biaya Software (R² = 0.88)', fontweight='bold')
axes[0].set_xlabel('Biaya Aktual (Juta IDR)')
axes[0].set_ylabel('Biaya Prediksi Model (Juta IDR)')
axes[0].legend()

# Plot Residual vs Nilai Prediksi (Uji Homoskedastisitas)
axes[1].scatter(df_cost['predicted_cost'], df_cost['residuals'], color='#2B6CB0', alpha=0.8)
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_title('Residual vs. Nilai Prediksi (Diagnostik Galat)', fontweight='bold')
axes[1].set_xlabel('Nilai Prediksi')
axes[1].set_ylabel('Residual Galat (e)')

plt.tight_layout()
plt.show()


## 📝 5. Kesimpulan Analisis & Data Storytelling

### ❓ Pertanyaan Refleksi & Konsep
* **Bagaimana cara menginterpretasikan koefisien $eta_1 = 3.82$ pada KLOC?** Artinya, setiap kenaikan 1.000 baris kode (1 KLOC) dengan asumsi variabel lain konstan (*ceteris paribus*), rata-rata biaya proyek akan meningkat sebesar **3.82 Juta IDR**.

### 🔍 Temuan Utama Data (Key Findings)
* Model menghasilkan nilai **$R^2 = 0.884$**, artinya **88.4% variasi biaya software** dapat dijelaskan secara akurat oleh ukuran kode, tim, dan durasi proyek.
* Uji simultan $F$-statistic membuktikan model prediktif sangat valid secara menyeluruh ($p < 0.0001$).

### 💡 Rekomendasi & Langkah Lanjutan
* Formula model ini dapat diintegrasikan ke dalam sistem estimasi anggaran (*COCOMO II Calculator*) untuk penawaran proyek software klien.
